In [ ]:
#%pip install pandas matplotlib seaborn sqlalchemy ipython-sql==0.4.1 prettytable==2.5.0
#%pip install tabulate  
#%pip install plotly
#%pip install nbformat>=4.2.0

# VDP Analysis
This notebook will be working with the 'VDP_dataset.csv' file, which was extracted from the Dune query at dune.com/queries/7717567. The table will have the following Columns:
 
| Column | Description |
|---|---|
| id | |
| name | |
| website | |
| auth_address | |
| current_commission | |
| total_stake | |
| vdp_stake | |
| Organic_stake | |
| initial_tier | |
| current_tier | |
| dependency_ratio | |
| Rewards_USD | |
| MonthlyReturns_USD | |
| EstimatedMonthlyReturns_USD | |
| underwater_status | |
| graduation_status | |

In [3]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///vdpdata.db')

csv_file = 'Vdp_dataset.csv'
table_name = 'vdp'
df = pd.read_csv(csv_file)
df.to_sql(table_name, engine, if_exists='replace', index=False)
print('successful')

successful


- How many validators received a delegation?
- Which delegation tier did each validator receive? 
- How concentrated is VDP stake among validators?


In [92]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go

query = """SELECT COUNT(DISTINCT id) as "Total VDP Validators"
FROM 'vdp'
WHERE initial_tier IS NOT NULL"""
df = pd.read_sql(query,engine)
total = df["Total VDP Validators"].iloc[0]
fig = go.Figure(
    go.Indicator(
        mode="number", 
        value=total, 
        number={
            "font": {
            "color": "purple",
            "family": "Inter"
        }},
        title={"text": "Total VDP Validators",
        "font": {"color": "Black"}}
        ))
fig.show()

In [38]:
import plotly.express as px

query = """SELECT initial_tier, COUNT(id) as total_delegators
FROM 'vdp'
WHERE initial_tier IS NOT NULL
GROUP BY initial_tier
ORDER BY count(id) desc"""
df = pd.read_sql(query, engine)
label = df["initial_tier"]
size = df["total_delegators"]
fig = px.pie(df, values=size, names=label, title="VDP Distribution Chart", hole=0.0)
fig.show()

In [37]:
import plotly.express as px

query = """SELECT current_tier, COUNT(id) as total_delegators
FROM 'vdp'
WHERE current_tier IS NOT NULL
GROUP BY current_tier
ORDER BY count(id) desc"""
df = pd.read_sql(query, engine)
label = df["current_tier"]
size = df["total_delegators"]
fig = px.pie(df, values=size, names=label, title="VDP Distribution Chart", hole=0.0)
fig.show()

In [93]:
query = """SELECT initial_tier, count(id) as total_delegators
FROM 'vdp'
WHERE initial_tier IS NOT NULL
GROUP BY initial_tier"""
df = pd.read_sql(query, engine)
fig = px.bar(df, x="initial_tier", y="total_delegators", title="VDP Distribution Chart")
#fig.update_layout(, showlegend=False, xaxis_title="Delegation Tier", yaxis="Number of Validators")
fig.show()

In [94]:
# How Concentrated is VDP Stake

query="""SELECT SUM(vdp_stake) * 1.0 / SUM(total_stake) * 1.0 AS concentration_ratio
FROM 'vdp'
"""
df = pd.read_sql(query,engine)
df

,concentration_ratio
0,0.738648


Herfindal-Hirschman Index (HHI)

In [ ]:
query ="""with shares AS (SELECT vdp_stake * 1.0 / SUM(vdp_stake) OVER() AS share
FROM vdp)
select sum(power(share, 2)) * 10000 AS hhi_10000
FROM shares"""
df = pd.read_sql(query, engine)
df

In [42]:
query="""SELECT dependency_ratio
FROM 'vdp'
"""
df = pd.read_sql(query,engine)
ratio = df["dependency_ratio"]
fig = px.histogram(df, x="dependency_ratio", nbins=5)
fig.show()

In [95]:
query ="""SELECT name, vdp_stake, organic_stake
FROM 'vdp'
"""
df = pd.read_sql(query, engine)
fig = px.scatter(df, y="vdp_stake", x="organic_stake", hover_name="name")
fig.show()

In [45]:
query = """SELECT name As "Validator Name", organic_stake As "Organic Stake"
FROM 'vdp'
ORDER BY organic_stake DESC
LIMIT 20"""

df = pd.read_sql(query,engine)
fig = px.bar(df, x="Validator Name", y="Organic Stake", title="Top 20 Validators by Organic Stake")
fig.show()

In [70]:
query = """
SELECT count(*) as "Total Validators", CASE WHEN graduation_status IS false THEN "undergraduated" else "graduated" 
end as "Status"
from 'vdp'
group by 2
"""
df = pd.read_sql(query,engine)
names = df["Status"]
values = df["Total Validators"]
fig = px.pie(df, names=names, values=values, title="Validator Graduation Status")
fig.show()

In [72]:
query = """
SELECT count(*) as "Total Validators", CASE WHEN underwater_status IS true THEN "Underwater" else "Above Water" 
end as "Status"
from 'vdp'
group by 2
"""
df = pd.read_sql(query,engine)
names = df["Status"]
values = df["Total Validators"]
fig = px.pie(df, names=names, values=values, title="Validator Sustainability")
fig.show()

In [77]:
query ="""SELECT name AS Validator, ROUND(MonthlyReturns_USD,2) AS "Current Monthly Returns", rOUND(EstimatedMonthlyReturns_USD,2) as "Estimated Returns Without VDP"
FROM 'vdp'
"""
df = pd.read_sql(query, engine)
fig = px.scatter(df, x="Current Monthly Returns", y="Estimated Returns Without VDP", hover_name="Validator")
fig.update_layout(yaxis_tickprefix = '$',
xaxis_tickprefix='$')
fig.show()

In [116]:
query = """
SELECT sum(organic_stake) as stake, 'organic' as type
from 'vdp'
union
select sum(vdp_stake) as stake, 'vdp' as type
from 'vdp'
"""
df = pd.read_sql(query,engine)
names = df["type"]
values = df["stake"].astype(int)
fig = px.pie(df, names=names, values=values, title="Stake by Type")
fig.update_traces(hovertemplate="<b>%{label}</b> <br>" + "Stake:%{value:,.0f} MON<br>"
+ "Share: %{percent}<extra></extra>")
fig.show()


In [127]:
query = """
WITH main AS (
    SELECT row_number() over(order by total_stake desc) as row, name, total_stake, organic_stake
    FROM vdp
    ORDER BY total_stake DESC
    LIMIT 50
),

T20 AS (
    SELECT SUM(total_stake) AS top_20_total, SUM(organic_stake) AS top_20_organic
    FROM main
    WHERE row <= 20
),

T10 AS (
    SELECT SUM(total_stake) AS top_10_total, SUM(organic_stake) AS top_10_organic
    FROM main
    WHERE row <= 10
),

T50 AS (
    SELECT SUM(total_stake) AS top_50_total, SUM(organic_stake) AS top_50_organic
    FROM main
    WHERE row <= 50
), 

totals AS (
    SELECT sum(total_stake) AS total, sum(organic_stake) AS organic
    FROM vdp
)

SELECT 'Top 10 Share' AS metric, 
       top_10_total / total AS Current, 
       top_10_organic / organic AS "Without VDP"
FROM totals, T10

UNION ALL

SELECT 'Top 20 Share' AS metric, 
       top_20_total / total AS Current, 
       top_20_organic / organic AS "Without VDP"
FROM totals, T20

UNION ALL

SELECT 'Top 50 Share' AS metric, 
       top_50_total / total AS Current, 
       top_50_organic / organic AS "Without VDP"
FROM totals, T50
"""

df = pd.read_sql(query, engine)
df['Current'] = df['Current'].map('{:.2%}'.format)
df['Without VDP'] = df['Without VDP'].map('{:.2%}'.format)

fig = go.Figure(
    data = [go.Table(
        header=dict(values=list(df.columns),
        align='left'),
        cells=dict(values=[df[col] for col in df.columns],
        align='left')
    )]
)
fig.show()